### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT *
FROM "Order" where order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

### Debug: check out df columns name

In [ ]:
df_order_completed_shipped.columns

### Statistics on sales revenue for 2023-2024 each month

In [ ]:
df_order_completed_shipped['order_date'] = pd.to_datetime(df_order_completed_shipped['order_date'])
df_order_completed_shipped['order_month'] = df_order_completed_shipped['order_date'].dt.to_period('M')

monthly_sales = (
    df_order_completed_shipped.groupby('order_month', as_index=False)['total_price_before_tax']
      .sum()
      .sort_values('order_month')
)

monthly_sales['order_month'] = monthly_sales['order_month'].dt.to_timestamp()
# 保存为 parquet 文件
monthly_sales.to_parquet('2023-2024_each_month_sales.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_each_month_sales.parquet 文件")

### Statistics on sales revenue for 2023-2024 each quarter

In [ ]:
df_order_completed_shipped['order_date'] = pd.to_datetime(df_order_completed_shipped['order_date'])
df_order_completed_shipped['order_quarter'] = df_order_completed_shipped['order_date'].dt.to_period('Q')

quarterly_sales = (
    df_order_completed_shipped.groupby('order_quarter', as_index=False)['total_price_before_tax']
      .sum()
      .sort_values('order_quarter')
)

quarterly_sales['order_quarter'] = quarterly_sales['order_quarter'].astype(str).str.replace('Q', '-Q')
quarterly_sales.to_parquet('2023-2024_each_quarter_sales.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_each_quarter_sales.parquet 文件")

### Statistics on sales revenue for 2023-2024 each year

In [ ]:
df_order_completed_shipped['order_year'] = df_order_completed_shipped['order_date'].dt.year

annual_sales = (
    df_order_completed_shipped.groupby('order_year', as_index=False)['total_price_before_tax']
      .sum()
      .sort_values('order_year')
)

annual_sales.to_parquet('2023-2024_each_year_sales.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_each_year_sales.parquet 文件")

In [ ]:
# 关闭数据库连接
engine.dispose()